# Baseline CICIoT2023 Without Feature Augmentation — DNN, Five-Seed Evaluation

In [1]:
# Global reproducibility setup
import os
import random
import numpy as np
import tensorflow as tf

SEED = 42
RANDOM_STATE = SEED
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("Global seed set to:", SEED)

Global seed set to: 42


In [2]:
from pathlib import Path
import gc
import json
import platform
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    multilabel_confusion_matrix,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [3]:
# ============================================================
# Load pre-generated CICIoT2023 partitions
# ============================================================
AUGMENTED_DATA_DIRECTORY = Path(
    r"/kaggle/input/datasets/zerukifle/ciciot2023-fa-multirun"
)

TRAIN_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "CICIoT2023_Augmented_Train.csv"
)
VALIDATION_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "CICIoT2023_Augmented_Validation.csv"
)
TEST_FILE = (
    AUGMENTED_DATA_DIRECTORY
    / "CICIoT2023_Augmented_Test.csv"
)

for file_path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    TEST_FILE,
]:
    if not file_path.is_file():
        raise FileNotFoundError(
            f"Required CICIoT2023 partition not found: {file_path}"
        )

train_data = pd.read_csv(TRAIN_FILE)
validation_data = pd.read_csv(VALIDATION_FILE)
test_data = pd.read_csv(TEST_FILE)

# This is the No-FA baseline, so remove the three anomaly-score features
# from all pre-generated partitions before the existing pipeline continues.
anomaly_score_columns = [
    "IF_Anomaly_Score",
    "AE_Anomaly_Score",
    "SUOD_Anomaly_Score",
]

for dataset_name, dataset in {
    "Training": train_data,
    "Validation": validation_data,
    "Test": test_data,
}.items():
    missing_anomaly_features = [
        feature
        for feature in anomaly_score_columns
        if feature not in dataset.columns
    ]

    if missing_anomaly_features:
        raise KeyError(
            f"{dataset_name} partition is missing anomaly-score features: "
            f"{missing_anomaly_features}"
        )

train_data = train_data.drop(columns=anomaly_score_columns)
validation_data = validation_data.drop(columns=anomaly_score_columns)
test_data = test_data.drop(columns=anomaly_score_columns)

# Retain the original variable for compatibility with the unchanged
# display/validation cells that follow.
df_CICIoT2023 = pd.concat(
    [train_data, validation_data, test_data],
    axis=0,
    ignore_index=True,
)

# Retained for compatibility with the unchanged experiment_config block.
INPUT_CSV = AUGMENTED_DATA_DIRECTORY


In [4]:
df_CICIoT2023.head()

,flow_duration,Header_Length,Protocol Type,Duration,Rate,Drate,fin_flag_number,syn_flag_number,psh_flag_number,ack_flag_number,...,IPv,Tot sum,Min,Max,AVG,Tot size,IAT,Covariance,Variance,label
0,0.244612,38791.0,17.00,64.00,3202.228500,0.0,0.0,0.0,0.0,0.0,...,1.0,525.00,50.0,50.00,50.00000,50.00,83102420.0,0.000000,0.00,DDoS
1,0.014093,5675.0,17.00,64.00,8785.439000,0.0,0.0,0.0,0.0,0.0,...,1.0,525.00,50.0,50.00,50.00000,50.00,83123740.0,0.000000,0.00,DDoS
2,0.000000,0.0,47.00,64.00,218.863710,0.0,0.0,0.0,0.0,0.0,...,1.0,6216.00,592.0,592.00,592.00000,592.00,83698550.0,0.000000,0.00,Mirai
3,0.000000,54.0,6.00,64.00,3.887012,0.0,0.0,0.0,0.0,0.0,...,1.0,567.00,54.0,54.00,54.00000,54.00,83064136.0,0.000000,0.00,DDoS
4,0.000000,0.0,0.99,63.36,5.231067,0.0,0.0,0.0,0.0,0.0,...,1.0,442.98,42.0,43.98,42.13838,42.18,83150650.0,2.315749,0.11,DDoS


In [5]:
df_CICIoT2023.shape

(591229, 33)

## 2.Separate labels and input features 

In [6]:
## 2. Feature-target separation and stratified data splitting
LABEL_COLUMN = "label"

if LABEL_COLUMN not in df_CICIoT2023.columns:
    raise KeyError(f"'{LABEL_COLUMN}' was not found in the dataset.")

if df_CICIoT2023[LABEL_COLUMN].isna().any():
    raise ValueError(f"'{LABEL_COLUMN}' contains missing values.")

feature_columns = [
    column for column in df_CICIoT2023.columns
    if column != LABEL_COLUMN
]
X = df_CICIoT2023[feature_columns].copy()
y = df_CICIoT2023[LABEL_COLUMN].astype(str).str.strip()

if y.eq("").any():
    raise ValueError(f"'{LABEL_COLUMN}' contains empty labels.")

non_numeric_columns = X.select_dtypes(exclude=np.number).columns.tolist()
if non_numeric_columns:
    raise TypeError(
        "Non-numeric input features were found: "
        f"{non_numeric_columns}"
    )

if not np.isfinite(X.to_numpy(dtype=np.float32)).all():
    raise ValueError("Input features contain missing or infinite values.")

print(f"Feature matrix: {X.shape}")
print(f"Target classes: {y.nunique()}")


Feature matrix: (591229, 32)
Target classes: 8


In [7]:
# Use the pre-generated train/validation/test partitions directly.
# No new train_test_split is performed here.
X_train = train_data[feature_columns].copy()
X_val = validation_data[feature_columns].copy()
X_test = test_data[feature_columns].copy()

y_train_original = (
    train_data[LABEL_COLUMN]
    .astype(str)
    .str.strip()
)
y_val_original = (
    validation_data[LABEL_COLUMN]
    .astype(str)
    .str.strip()
)
y_test_original = (
    test_data[LABEL_COLUMN]
    .astype(str)
    .str.strip()
)

for dataset_name, dataset in {
    "Validation": validation_data,
    "Test": test_data,
}.items():
    current_features = [
        column for column in dataset.columns
        if column != LABEL_COLUMN
    ]
    if current_features != feature_columns:
        raise ValueError(
            f"{dataset_name} feature columns/order do not match training."
        )

# Preserve row positions within each pre-generated partition for the
# existing reusable-artifact structure.
train_indices = X_train.index.to_numpy(copy=True)
validation_indices = X_val.index.to_numpy(copy=True)
test_indices = X_test.index.to_numpy(copy=True)

X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train_original = y_train_original.reset_index(drop=True)
y_val_original = y_val_original.reset_index(drop=True)
y_test_original = y_test_original.reset_index(drop=True)

# Fit label encoding on training labels and reuse the mapping.
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_original)

training_classes = set(label_encoder.classes_)
unknown_validation_classes = set(y_val_original.unique()) - training_classes
unknown_test_classes = set(y_test_original.unique()) - training_classes
if unknown_validation_classes or unknown_test_classes:
    raise ValueError(
        "Validation/test partitions contain classes absent from training."
    )

y_val = label_encoder.transform(y_val_original)
y_test = label_encoder.transform(y_test_original)
num_classes = len(label_encoder.classes_)

print("Multiclass label mapping:")
for class_code, class_name in enumerate(label_encoder.classes_):
    print(f"{class_code}: {class_name}")

# Raw partition DataFrames are no longer required after extraction.
del train_data, validation_data, test_data


Multiclass label mapping:
0: Benign
1: BruteForce
2: DDoS
3: DoS
4: Mirai
5: Recon
6: Spoofing
7: Web


In [8]:
print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Test set:       {X_test.shape}")
print(f"Number of training classes: {len(np.unique(y_train))}")

Training set:   (354737, 32)
Validation set: (118246, 32)
Test set:       (118246, 32)
Number of training classes: 8


In [9]:
def display_class_distribution(encoded_labels, partition_name):
    distribution = (
        pd.Series(encoded_labels)
        .value_counts(normalize=True)
        .sort_index()
        .rename(index=dict(enumerate(label_encoder.classes_)))
        .apply(lambda value: f"{value:.2%}")
    )
    print(f"\n{partition_name}:")
    print(distribution)

print("Class distribution (normalized frequency):")
display_class_distribution(y_train, "Training set")
display_class_distribution(y_val, "Validation set")
display_class_distribution(y_test, "Test set")

Class distribution (normalized frequency):

Training set:
Benign         2.33%
BruteForce     0.03%
DDoS          72.76%
DoS           17.31%
Mirai          5.70%
Recon          0.77%
Spoofing       1.05%
Web            0.05%
Name: proportion, dtype: object

Validation set:
Benign         2.33%
BruteForce     0.03%
DDoS          72.76%
DoS           17.31%
Mirai          5.70%
Recon          0.77%
Spoofing       1.05%
Web            0.05%
Name: proportion, dtype: object

Test set:
Benign         2.32%
BruteForce     0.03%
DDoS          72.76%
DoS           17.31%
Mirai          5.70%
Recon          0.77%
Spoofing       1.05%
Web            0.05%
Name: proportion, dtype: object


In [10]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape ,X_val.shape,y_val.shape

((354737, 32), (118246, 32), (354737,), (118246,), (118246, 32), (118246,))

In [11]:
y_count = pd.Series(y)
y_count.value_counts()

label
DDoS          430167
DoS           102342
Mirai          33719
Benign         13747
Spoofing        6215
Recon           4560
Web              313
BruteForce       166
Name: count, dtype: int64

In [12]:
y_train_count = (
    pd.Series(y_train).value_counts().sort_index()
    .rename(index=dict(enumerate(label_encoder.classes_)))
)
y_train_count

Benign          8248
BruteForce       100
DDoS          258100
DoS            61405
Mirai          20231
Recon           2736
Spoofing        3729
Web              188
Name: count, dtype: int64

In [13]:
y_test_count = (
    pd.Series(y_test).value_counts().sort_index()
    .rename(index=dict(enumerate(label_encoder.classes_)))
)
y_test_count

Benign         2749
BruteForce       33
DDoS          86034
DoS           20469
Mirai          6744
Recon           912
Spoofing       1243
Web              62
Name: count, dtype: int64

##### Validation-set class counts

In [14]:
y_val_count = (
    pd.Series(y_val).value_counts().sort_index()
    .rename(index=dict(enumerate(label_encoder.classes_)))
)
y_val_count

Benign         2750
BruteForce       33
DDoS          86033
DoS           20468
Mirai          6744
Recon           912
Spoofing       1243
Web              63
Name: count, dtype: int64

In [15]:
X_val.columns , X_val.shape

(Index(['flow_duration', 'Header_Length', 'Protocol Type', 'Duration', 'Rate',
        'Drate', 'fin_flag_number', 'syn_flag_number', 'psh_flag_number',
        'ack_flag_number', 'syn_count', 'fin_count', 'urg_count', 'rst_count',
        'HTTP', 'HTTPS', 'DNS', 'SSH', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP',
        'IPv', 'Tot sum', 'Min', 'Max', 'AVG', 'Tot size', 'IAT', 'Covariance',
        'Variance'],
       dtype='object'),
 (118246, 32))

### Standardize features 

In [16]:
# Fit only on training features, then apply the fitted scaler unchanged.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

for partition_name, values in {
    "training": X_train_scaled,
    "validation": X_val_scaled,
    "test": X_test_scaled,
}.items():
    if not np.isfinite(values).all():
        raise ValueError(f"Scaled {partition_name} features are invalid.")

print(f"Scaled training features:   {X_train_scaled.shape}")
print(f"Scaled validation features: {X_val_scaled.shape}")
print(f"Scaled test features:       {X_test_scaled.shape}")

# The unscaled partitions are no longer required in memory.
del X_train, X_val, X_test

Scaled training features:   (354737, 32)
Scaled validation features: (118246, 32)
Scaled test features:       (118246, 32)


## 3. Five-run baseline configuration and reusable artifacts


In [17]:
# Fixed repeated-training and inference-timing protocol
RUN_SEEDS = [42, 43, 44, 45, 46]
MODEL_TYPE = "DNN"
N_WARMUP_RUNS = 3
N_BENCHMARK_RUNS = 5

# Fixed baseline settings from the uploaded source notebook.
BASELINE_HYPERPARAMETERS = {
    "hidden_units_1": 64,
    "hidden_units_2": 32,
    "dropout_rate": 0.2,
    "learning_rate": 0.001,
    "l2_regularization": 1e-4,
    "epochs": 100,
    "batch_size": 64,
    "shuffle": True,
    "early_stopping": {
        "monitor": "val_loss",
        "patience": 10,
        "min_delta": 0.0,
        "restore_best_weights": True,
    },
}

OUTPUT_DIRECTORY = Path.cwd() / "CICIoT2023_baseline_without_FA_dnn_5_seed_results"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIRECTORY = OUTPUT_DIRECTORY / "reusable_artifacts"
ARTIFACT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Save fixed preprocessing objects once.
joblib.dump(scaler, ARTIFACT_DIRECTORY / "standard_scaler.joblib")
joblib.dump(label_encoder, ARTIFACT_DIRECTORY / "label_encoder.joblib")
with (ARTIFACT_DIRECTORY / "feature_columns.json").open("w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)
np.savez_compressed(
    ARTIFACT_DIRECTORY / "split_indices.npz",
    train=train_indices,
    validation=validation_indices,
    test=test_indices,
)

experiment_config = {
    "model": MODEL_TYPE,
    "dataset_path": str(INPUT_CSV),
    "feature_augmentation": False,
    "optuna": False,
    "split_random_state": int(RANDOM_STATE),
    "split_proportions": {"training": 0.60, "validation": 0.20, "test": 0.20},
    "run_seeds": RUN_SEEDS,
    "hyperparameters": BASELINE_HYPERPARAMETERS,
    "inference_protocol": {
        "warmup_runs": N_WARMUP_RUNS,
        "measured_runs": N_BENCHMARK_RUNS,
    },
}
with (OUTPUT_DIRECTORY / "experiment_config.json").open("w", encoding="utf-8") as f:
    json.dump(experiment_config, f, indent=2)

print("Model:", MODEL_TYPE)
print("Run seeds:", RUN_SEEDS)
print("Output directory:", OUTPUT_DIRECTORY)


Model: DNN
Run seeds: [42, 43, 44, 45, 46]
Output directory: /kaggle/working/CICIoT2023_baseline_without_FA_dnn_5_seed_results


## 4. Reproducibility, metrics, timing, and reporting helpers


In [18]:
def set_run_seed(seed):
    """Reset model-training randomness for one independent run."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)


try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except (AttributeError, RuntimeError):
    print("TensorFlow deterministic-operation control is unavailable.")


def benchmark_full_test_prediction(model, X_input):
    """Time complete-test prediction using 3 warm-ups + 10 measured runs."""
    warmup_size = min(1024, len(X_input))
    for _ in range(N_WARMUP_RUNS):
        model.predict(X_input[:warmup_size], verbose=0)

    measured_times = []
    probabilities = None
    for _ in range(N_BENCHMARK_RUNS):
        start_time = time.perf_counter()
        probabilities = model.predict(X_input, verbose=0)
        measured_times.append(time.perf_counter() - start_time)

    mean_inference_time = float(np.mean(measured_times))
    return probabilities, {
        "Inference_Time_s": mean_inference_time,
        "Latency_ms_per_sample": mean_inference_time / len(X_input) * 1000.0,
        "Throughput_samples_per_s": len(X_input) / mean_inference_time,
    }


def calculate_fpr_details(y_true, y_pred, labels):
    matrices = multilabel_confusion_matrix(y_true, y_pred, labels=labels)
    rows = []
    for class_index, matrix in enumerate(matrices):
        tn, fp, fn, tp = matrix.ravel()
        denominator = fp + tn
        fpr = fp / denominator if denominator else 0.0
        rows.append({
            "Class_Index": int(class_index),
            "TP": int(tp),
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "FPR": float(fpr),
            "Support": int((y_true == labels[class_index]).sum()),
        })
    frame = pd.DataFrame(rows)
    micro_denominator = frame["FP"].sum() + frame["TN"].sum()
    summary = {
        "FPR_macro": float(frame["FPR"].mean()),
        "FPR_micro_overall": float(
            frame["FP"].sum() / micro_denominator if micro_denominator else 0.0
        ),
        "FPR_weighted": float(np.average(frame["FPR"], weights=frame["Support"])),
    }
    return frame, summary


def calculate_test_metrics(y_true, y_pred, probabilities, labels):
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision_macro": precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "Recall_macro": recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "F1_macro": f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0),
        "Precision_weighted": precision_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "Recall_weighted": recall_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "F1_weighted": f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
    try:
        metrics["ROC_AUC_macro_OvR"] = roc_auc_score(
            y_true, probabilities, labels=labels, average="macro", multi_class="ovr"
        )
    except ValueError as error:
        metrics["ROC_AUC_macro_OvR"] = np.nan
        print(f"Macro ROC-AUC could not be calculated: {error}")

    fpr_details, fpr_summary = calculate_fpr_details(y_true, y_pred, labels)
    metrics.update(fpr_summary)
    return {key: float(value) for key, value in metrics.items()}, fpr_details


def format_mean_sd(mean_value, sd_value, decimals=6, percentage=False):
    scale = 100.0 if percentage else 1.0
    if pd.isna(mean_value):
        return "NA"
    if pd.isna(sd_value):
        return f"{mean_value * scale:.{decimals}f} ± NA"
    return f"{mean_value * scale:.{decimals}f} ± {sd_value * scale:.{decimals}f}"


def save_confusion_matrix_figure(matrix, output_stem):
    """Save report-style confusion matrix with both axes rotated 45 degrees."""
    plt.figure(figsize=(6.8, 5.5))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=label_encoder.classes_,
        yticklabels=label_encoder.classes_,
    )
    plt.xlabel("Predicted label", fontsize=11)
    plt.ylabel("True label", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=10)
    plt.yticks(rotation=45, va="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{output_stem}.pdf", bbox_inches="tight")
    plt.savefig(f"{output_stem}.png", dpi=600, bbox_inches="tight")
    plt.close()
def save_loss_curve(history_data, output_stem, title=None):
    """Save training/validation loss curve as PDF and 600-dpi PNG."""
    if isinstance(history_data, pd.DataFrame):
        history_frame = history_data.copy()
    else:
        history_frame = pd.DataFrame(history_data)

    if "loss" not in history_frame.columns:
        raise KeyError("The training history does not contain a 'loss' column.")

    epochs = np.arange(1, len(history_frame) + 1)

    plt.figure(figsize=(6.8, 5.0))
    plt.plot(epochs, history_frame["loss"], label="Training Loss")

    if "val_loss" in history_frame.columns:
        plt.plot(epochs, history_frame["val_loss"], label="Validation Loss")

    plt.xlabel("Epoch", fontsize=11)
    plt.ylabel("Loss", fontsize=11)
    if title:
        plt.title(title, fontsize=11)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{output_stem}.pdf", bbox_inches="tight")
    plt.savefig(f"{output_stem}.png", dpi=600, bbox_inches="tight")
    plt.close()


TensorFlow deterministic operations enabled.


## 5. Fixed DNN baseline model


In [19]:
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

num_features = X_train_scaled.shape[1]


def build_final_dnn():
    parameters = BASELINE_HYPERPARAMETERS
    regularizer = regularizers.l2(parameters["l2_regularization"])
    model = Sequential(name="Baseline_DNN")
    model.add(Input(shape=(num_features,)))
    model.add(Dense(parameters["hidden_units_1"], activation="relu", kernel_regularizer=regularizer))
    model.add(Dropout(parameters["dropout_rate"]))
    model.add(Dense(parameters["hidden_units_2"], activation="relu", kernel_regularizer=regularizer))
    model.add(Dropout(parameters["dropout_rate"]))
    model.add(Dense(num_classes, activation="softmax", kernel_regularizer=regularizer))
    model.compile(
        optimizer=Adam(learning_rate=parameters["learning_rate"]),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def make_early_stopping():
    settings = BASELINE_HYPERPARAMETERS["early_stopping"]
    return EarlyStopping(
        monitor=settings["monitor"],
        patience=settings["patience"],
        min_delta=settings["min_delta"],
        restore_best_weights=settings["restore_best_weights"],
        verbose=1,
    )

X_train_input = X_train_scaled
X_val_input = X_val_scaled
X_test_input = X_test_scaled


## 6. Execute five independent training runs 


In [20]:
labels = np.arange(num_classes)
run_records = []
per_class_records = []
macro_records = []
weighted_records = []
representative_candidates = []

for run_number, seed in enumerate(RUN_SEEDS, start=1):
    print("\n" + "=" * 72)
    print(f"{MODEL_TYPE}: run {run_number}/{len(RUN_SEEDS)}; seed={seed}")
    print("=" * 72)

    seed_directory = OUTPUT_DIRECTORY / f"{MODEL_TYPE}_Seed_{run_number}"
    seed_directory.mkdir(parents=True, exist_ok=True)

    tf.keras.backend.clear_session()
    set_run_seed(seed)
    model = build_final_dnn()

    training_start = time.perf_counter()
    history_object = model.fit(
        X_train_input,
        y_train,
        validation_data=(X_val_input, y_val),
        epochs=BASELINE_HYPERPARAMETERS["epochs"],
        batch_size=BASELINE_HYPERPARAMETERS["batch_size"],
        callbacks=[make_early_stopping()],
        shuffle=BASELINE_HYPERPARAMETERS["shuffle"],
        verbose=1,
    )
    training_time = time.perf_counter() - training_start

    validation_probabilities = model.predict(X_val_input, verbose=0)
    validation_predictions = np.argmax(validation_probabilities, axis=1)
    validation_macro_f1 = f1_score(
        y_val, validation_predictions, labels=labels, average="macro", zero_division=0
    )

    evaluation = model.evaluate(X_test_input, y_test, verbose=0, return_dict=True)
    test_probabilities, timing_metrics = benchmark_full_test_prediction(model, X_test_input)
    test_predictions = np.argmax(test_probabilities, axis=1)
    test_metrics, fpr_details = calculate_test_metrics(
        y_test, test_predictions, test_probabilities, labels
    )

    run_record = {
        "Model": MODEL_TYPE,
        "Run": int(run_number),
        "Seed": int(seed),
        "Epochs_Trained": int(len(history_object.history["loss"])),
        "Training_Time_s": float(training_time),
        "Validation_F1_macro": float(validation_macro_f1),
        "Test_Loss": float(evaluation["loss"]),
        **test_metrics,
        **timing_metrics,
    }
    run_records.append(run_record)

    # Separate macro report for this run.
    macro_record = {
        "Model": MODEL_TYPE,
        "Run": int(run_number),
        "Seed": int(seed),
        "Precision_macro": run_record["Precision_macro"],
        "Recall_macro": run_record["Recall_macro"],
        "F1_macro": run_record["F1_macro"],
        "ROC_AUC_macro_OvR": run_record["ROC_AUC_macro_OvR"],
        "FPR_macro": run_record["FPR_macro"],
    }
    macro_records.append(macro_record)

    # Separate weighted-average report for this run.
    weighted_record = {
        "Model": MODEL_TYPE,
        "Run": int(run_number),
        "Seed": int(seed),
        "Precision_weighted": run_record["Precision_weighted"],
        "Recall_weighted": run_record["Recall_weighted"],
        "F1_weighted": run_record["F1_weighted"],
        "FPR_weighted": run_record["FPR_weighted"],
    }
    weighted_records.append(weighted_record)

    precision_values, recall_values, f1_values, support_values = precision_recall_fscore_support(
        y_test, test_predictions, labels=labels, zero_division=0
    )
    per_class_frame = fpr_details.copy()
    per_class_frame["Model"] = MODEL_TYPE
    per_class_frame["Run"] = int(run_number)
    per_class_frame["Seed"] = int(seed)
    per_class_frame["Class"] = label_encoder.classes_
    per_class_frame["Precision"] = precision_values
    per_class_frame["Recall"] = recall_values
    per_class_frame["F1"] = f1_values
    per_class_frame["Support"] = support_values.astype(int)
    per_class_frame = per_class_frame[[
        "Model", "Run", "Seed", "Class", "Precision", "Recall", "F1", "FPR",
        "TP", "TN", "FP", "FN", "Support"
    ]]
    per_class_records.extend(per_class_frame.to_dict(orient="records"))

    # Save per-seed scalar and separate macro/weighted reports.
    pd.DataFrame([run_record]).to_csv(seed_directory / "scalar_metrics.csv", index=False)
    pd.DataFrame([macro_record]).to_csv(seed_directory / "macro_metrics.csv", index=False)
    pd.DataFrame([weighted_record]).to_csv(seed_directory / "weighted_average_metrics.csv", index=False)
    per_class_frame.to_csv(seed_directory / "per_class_metrics_and_fpr.csv", index=False)

    fpr_summary_frame = pd.DataFrame([{
        "FPR_macro": run_record["FPR_macro"],
        "FPR_micro_overall": run_record["FPR_micro_overall"],
        "FPR_weighted": run_record["FPR_weighted"],
    }])
    fpr_summary_frame.to_csv(seed_directory / "false_positive_rate_summary.csv", index=False)

    report = classification_report(
        y_test,
        test_predictions,
        labels=labels,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    pd.DataFrame(report).transpose().to_csv(seed_directory / "classification_report.csv")

    matrix = confusion_matrix(y_test, test_predictions, labels=labels)
    pd.DataFrame(matrix, index=label_encoder.classes_, columns=label_encoder.classes_).to_csv(
        seed_directory / "confusion_matrix.csv"
    )
    save_confusion_matrix_figure(matrix, seed_directory / "confusion_matrix")

    history = {
        key: [float(value) for value in values]
        for key, values in history_object.history.items()
    }
    history_frame = pd.DataFrame(history)
    history_frame.to_csv(
        seed_directory / "training_history.csv",
        index=False,
    )
    save_loss_curve(
        history_frame,
        seed_directory / "loss_curve",
    )

    prediction_frame = pd.DataFrame({
        "True_Code": y_test,
        "True_Label": label_encoder.inverse_transform(y_test),
        "Predicted_Code": test_predictions,
        "Predicted_Label": label_encoder.inverse_transform(test_predictions),
    })
    for class_index, class_name in enumerate(label_encoder.classes_):
        prediction_frame[f"Probability_{class_name}"] = test_probabilities[:, class_index]
    prediction_frame.to_csv(seed_directory / "test_predictions_and_probabilities.csv", index=False)

    model_path = seed_directory / f"{MODEL_TYPE.lower()}_seed_{seed}.keras"
    model.save(model_path)

    run_metadata = {
        "Model": MODEL_TYPE,
        "Run": int(run_number),
        "Seed": int(seed),
        "Feature_Augmentation": False,
        "Optuna": False,
        "Split_Random_State": int(RANDOM_STATE),
        "Hyperparameters": BASELINE_HYPERPARAMETERS,
        "Inference_Warmups": N_WARMUP_RUNS,
        "Inference_Measured_Runs": N_BENCHMARK_RUNS,
    }
    with (seed_directory / "run_metadata.json").open("w", encoding="utf-8") as f:
        json.dump(run_metadata, f, indent=2)

    representative_candidates.append({
        "Seed": int(seed),
        "Run": int(run_number),
        "F1_macro": float(run_record["F1_macro"]),
        "Predictions": test_predictions.copy(),
        "Model_Path": str(model_path),
    })

    print(
        f"Completed {MODEL_TYPE} seed {seed}: "
        f"macro F1={run_record['F1_macro']:.6f}, "
        f"weighted F1={run_record['F1_weighted']:.6f}"
    )

    del model, history_object, validation_probabilities, test_probabilities
    gc.collect()
    tf.keras.backend.clear_session()

all_run_results = pd.DataFrame(run_records)
all_per_class_results = pd.DataFrame(per_class_records)
all_macro_results = pd.DataFrame(macro_records)
all_weighted_results = pd.DataFrame(weighted_records)

all_run_results.to_csv(OUTPUT_DIRECTORY / "dnn_5_seed_run_results.csv", index=False)
all_per_class_results.to_csv(OUTPUT_DIRECTORY / "dnn_per_class_5_seed_run_results.csv", index=False)
all_macro_results.to_csv(OUTPUT_DIRECTORY / "dnn_macro_metrics_5_seed_results.csv", index=False)
all_weighted_results.to_csv(OUTPUT_DIRECTORY / "dnn_weighted_average_metrics_5_seed_results.csv", index=False)

display(all_run_results)



DNN: run 1/5; seed=42


2026-10-01 10:42:04.081932: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/100


2026-10-01 10:42:04.404633: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


5526/5543 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8019 - loss: 0.4920

...

2026-10-01 11:24:08.645559: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


5543/5543 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.8250 - loss: 0.4005 - val_accuracy: 0.8396 - val_loss: 0.3476
Epoch 2/100
5543/5543 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.8387 - loss: 0.3559 - val_accuracy: 0.8445 - val_loss: 0.3329
Epoch 3/100
5543/5543 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.8450 - loss: 0.3430 - val_accuracy: 0.8532 - val_loss: 0.3181
Epoch 4/100
5543/5543 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.8747 - loss: 0.3041 - val_accuracy: 0.9586 - val_loss: 0.1860
Epoch 5/100
5543/5543 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.9544 - loss: 0.1807 - val_accuracy: 0.9857 - val_loss: 0.1059
Epoch 6/100
5543/5543 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.9695 - loss: 0.1546 - val_accuracy: 0.9866 - val_loss: 0.0976
Epoch 7/100
5543/5543 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.9712 - loss: 0.1512 - val_accuracy: 0.9867 - val_loss: 0.0984
Epoch 8/100
5543/5543 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9721 - loss: 0.146

2026-10-01 11:39:12.356495: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2026-10-01 11:39:18.256105: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Completed DNN seed 46: macro F1=0.670067, weighted F1=0.986901


,Model,Run,Seed,Epochs_Trained,Training_Time_s,Validation_F1_macro,Test_Loss,Accuracy,Precision_macro,Recall_macro,...,Recall_weighted,F1_weighted,MCC,ROC_AUC_macro_OvR,FPR_macro,FPR_micro_overall,FPR_weighted,Inference_Time_s,Latency_ms_per_sample,Throughput_samples_per_s
0,DNN,1,42,37,644.596759,0.671401,0.094430,0.987052,0.799939,0.635308,...,0.987052,0.985981,0.970406,0.996986,0.002253,0.001850,0.005080,6.323054,0.053474,18700.774030
1,DNN,2,43,42,751.949704,0.635106,0.095258,0.987086,0.671151,0.623948,...,0.987086,0.986061,0.970518,0.996936,0.002109,0.001845,0.003954,5.544959,0.046893,21324.956244
2,DNN,3,44,24,435.780859,0.680899,0.094204,0.987238,0.790236,0.641876,...,0.987238,0.986274,0.970894,0.996406,0.001870,0.001823,0.002198,5.376816,0.045471,21991.826767
3,DNN,4,45,27,486.398507,0.680590,0.092718,0.987780,0.795364,0.643476,...,0.987780,0.986829,0.972096,0.996829,0.001912,0.001746,0.003073,5.551884,0.046952,21298.357030
4,DNN,5,46,51,920.863766,0.685229,0.094526,0.987797,0.788714,0.648086,...,0.987797,0.986901,0.972117,0.997204,0.001917,0.001743,0.003134,5.344725,0.045200,22123.871142


## 7. Five-run mean ± sample-SD summaries and representative results


In [21]:
identifier_columns = {"Model", "Run", "Seed"}
metric_columns = [column for column in all_run_results.columns if column not in identifier_columns]

summary_records = []
for metric_name in metric_columns:
    values = all_run_results[metric_name].astype(float)
    mean_value = float(values.mean())
    sd_value = float(values.std(ddof=1))
    summary_records.append({
        "Model": MODEL_TYPE,
        "Metric": metric_name,
        "Mean": mean_value,
        "SD": sd_value,
        "N": int(values.notna().sum()),
        "Mean ± SD": format_mean_sd(mean_value, sd_value),
    })

five_run_summary_long = pd.DataFrame(summary_records)
five_run_summary_long.to_csv(OUTPUT_DIRECTORY / "five_run_mean_sd_long.csv", index=False)
five_run_summary_long[["Metric", "Mean ± SD"]].to_csv(
    OUTPUT_DIRECTORY / "five_run_mean_sd_report_table.csv", index=False
)


def build_subset_summary(frame, metric_columns):
    rows = []
    for metric in metric_columns:
        values = frame[metric].astype(float)
        mean_value = float(values.mean())
        sd_value = float(values.std(ddof=1))
        rows.append({
            "Model": MODEL_TYPE,
            "Metric": metric,
            "Mean": mean_value,
            "SD": sd_value,
            "N": int(values.notna().sum()),
            "Mean ± SD": format_mean_sd(mean_value, sd_value),
        })
    return pd.DataFrame(rows)

macro_summary = build_subset_summary(
    all_macro_results,
    ["Precision_macro", "Recall_macro", "F1_macro", "ROC_AUC_macro_OvR", "FPR_macro"],
)
weighted_summary = build_subset_summary(
    all_weighted_results,
    ["Precision_weighted", "Recall_weighted", "F1_weighted", "FPR_weighted"],
)
macro_summary.to_csv(OUTPUT_DIRECTORY / "macro_metrics_5_seed_mean_sd.csv", index=False)
weighted_summary.to_csv(OUTPUT_DIRECTORY / "weighted_average_metrics_5_seed_mean_sd.csv", index=False)

# Compact publication tables. Accuracy is shown in both for convenient comparison.
accuracy_mean = float(all_run_results["Accuracy"].mean())
accuracy_sd = float(all_run_results["Accuracy"].std(ddof=1))

journal_macro = pd.DataFrame([{
    "Model": MODEL_TYPE,
    "Accuracy (%)": format_mean_sd(accuracy_mean, accuracy_sd, decimals=2, percentage=True),
    "Recall (macro %)": format_mean_sd(all_run_results["Recall_macro"].mean(), all_run_results["Recall_macro"].std(ddof=1), decimals=2, percentage=True),
    "Precision (macro %)": format_mean_sd(all_run_results["Precision_macro"].mean(), all_run_results["Precision_macro"].std(ddof=1), decimals=2, percentage=True),
    "F1-score (macro %)": format_mean_sd(all_run_results["F1_macro"].mean(), all_run_results["F1_macro"].std(ddof=1), decimals=2, percentage=True),
    "FPR (macro %)": format_mean_sd(all_run_results["FPR_macro"].mean(), all_run_results["FPR_macro"].std(ddof=1), decimals=4, percentage=True),
    "ROC-AUC (macro, OvR)": format_mean_sd(all_run_results["ROC_AUC_macro_OvR"].mean(), all_run_results["ROC_AUC_macro_OvR"].std(ddof=1), decimals=6),
}])

journal_weighted = pd.DataFrame([{
    "Model": MODEL_TYPE,
    "Accuracy (%)": format_mean_sd(accuracy_mean, accuracy_sd, decimals=2, percentage=True),
    "Recall (weighted %)": format_mean_sd(all_run_results["Recall_weighted"].mean(), all_run_results["Recall_weighted"].std(ddof=1), decimals=2, percentage=True),
    "Precision (weighted %)": format_mean_sd(all_run_results["Precision_weighted"].mean(), all_run_results["Precision_weighted"].std(ddof=1), decimals=2, percentage=True),
    "F1-score (weighted %)": format_mean_sd(all_run_results["F1_weighted"].mean(), all_run_results["F1_weighted"].std(ddof=1), decimals=2, percentage=True),
    "FPR (weighted %)": format_mean_sd(all_run_results["FPR_weighted"].mean(), all_run_results["FPR_weighted"].std(ddof=1), decimals=4, percentage=True),
}])

journal_macro.to_csv(OUTPUT_DIRECTORY / "journal_macro_mean_sd_table.csv", index=False)
journal_weighted.to_csv(OUTPUT_DIRECTORY / "journal_weighted_mean_sd_table.csv", index=False)

# Per-class five-run mean ± SD.
per_class_summary_records = []
for class_name, class_rows in all_per_class_results.groupby("Class", sort=False):
    record = {
        "Model": MODEL_TYPE,
        "Class": class_name,
        "Support": int(class_rows["Support"].iloc[0]),
        "N": int(len(class_rows)),
    }
    for metric in ["Precision", "Recall", "F1", "FPR"]:
        mean_value = float(class_rows[metric].mean())
        sd_value = float(class_rows[metric].std(ddof=1))
        record[f"{metric}_Mean"] = mean_value
        record[f"{metric}_SD"] = sd_value
        record[f"{metric}_Mean ± SD"] = format_mean_sd(mean_value, sd_value)
    per_class_summary_records.append(record)

per_class_mean_sd = pd.DataFrame(per_class_summary_records)
per_class_mean_sd.to_csv(OUTPUT_DIRECTORY / "per_class_five_run_mean_sd.csv", index=False)

# Representative run: macro F1 closest to the five-run mean.
mean_macro_f1 = float(all_run_results["F1_macro"].mean())
representative = min(
    representative_candidates,
    key=lambda item: abs(item["F1_macro"] - mean_macro_f1),
)
representative_seed = int(representative["Seed"])
representative_predictions = representative["Predictions"]
representative_matrix = confusion_matrix(y_test, representative_predictions, labels=labels)

pd.DataFrame(
    representative_matrix,
    index=label_encoder.classes_,
    columns=label_encoder.classes_,
).to_csv(OUTPUT_DIRECTORY / "dnn_representative_confusion_matrix.csv")
save_confusion_matrix_figure(
    representative_matrix,
    OUTPUT_DIRECTORY / "dnn_representative_confusion_matrix",
)

representative_report = classification_report(
    y_test,
    representative_predictions,
    labels=labels,
    target_names=label_encoder.classes_,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(representative_report).transpose().to_csv(
    OUTPUT_DIRECTORY / "dnn_representative_classification_report.csv"
)

representative_runs = pd.DataFrame([{
    "Model": MODEL_TYPE,
    "Representative_Seed": representative_seed,
    "Representative_F1_macro": float(representative["F1_macro"]),
    "Five_Run_Mean_F1_macro": mean_macro_f1,
    "Model_Path": representative["Model_Path"],
}])
representative_runs.to_csv(OUTPUT_DIRECTORY / "representative_runs.csv", index=False)

representative_run_number = int(representative["Run"])
representative_seed_directory = (
    OUTPUT_DIRECTORY
    / f"{MODEL_TYPE}_Seed_{representative_run_number}"
)
representative_history_path = (
    representative_seed_directory / "training_history.csv"
)

if representative_history_path.exists():
    representative_history_frame = pd.read_csv(
        representative_history_path
    )
    representative_history_frame.to_csv(
        OUTPUT_DIRECTORY
        / "dnn_representative_training_history.csv",
        index=False,
    )
    save_loss_curve(
        representative_history_frame,
        OUTPUT_DIRECTORY
        / "dnn_representative_loss_curve",
    )
else:
    print(
        "Representative training history was not found:",
        representative_history_path,
    )


print("Five-run summary:")
display(five_run_summary_long)
print("Macro mean ± SD:")
display(macro_summary)
print("Weighted-average mean ± SD:")
display(weighted_summary)
print("Journal macro table:")
display(journal_macro)
print("Journal weighted table:")
display(journal_weighted)
print("Representative run:")
display(representative_runs)
print("All results saved to:", OUTPUT_DIRECTORY)


Five-run summary:


,Model,Metric,Mean,SD,N,Mean ± SD
0,DNN,Epochs_Trained,36.200000,11.031772,5,36.200000 ± 11.031772
1,DNN,Training_Time_s,647.917919,197.752200,5,647.917919 ± 197.752200
2,DNN,Validation_F1_macro,0.670645,0.020496,5,0.670645 ± 0.020496
3,DNN,Test_Loss,0.094227,0.000932,5,0.094227 ± 0.000932
4,DNN,Accuracy,0.987391,0.000370,5,0.987391 ± 0.000370
5,DNN,Precision_macro,0.769081,0.054923,5,0.769081 ± 0.054923
6,DNN,Recall_macro,0.638539,0.009354,5,0.638539 ± 0.009354
7,DNN,F1_macro,0.661224,0.014680,5,0.661224 ± 0.014680
8,DNN,Precision_weighted,0.987752,0.000310,5,0.987752 ± 0.000310
9,DNN,Recall_weighted,0.987391,0.000370,5,0.987391 ± 0.000370


Macro mean ± SD:


,Model,Metric,Mean,SD,N,Mean ± SD
0,DNN,Precision_macro,0.769081,0.054923,5,0.769081 ± 0.054923
1,DNN,Recall_macro,0.638539,0.009354,5,0.638539 ± 0.009354
2,DNN,F1_macro,0.661224,0.014680,5,0.661224 ± 0.014680
3,DNN,ROC_AUC_macro_OvR,0.996872,0.000294,5,0.996872 ± 0.000294
4,DNN,FPR_macro,0.002012,0.000163,5,0.002012 ± 0.000163


Weighted-average mean ± SD:


...

Journal weighted table:


,Model,Accuracy (%),Recall (weighted %),Precision (weighted %),F1-score (weighted %),FPR (weighted %)
0,DNN,98.74 ± 0.04,98.74 ± 0.04,98.78 ± 0.03,98.64 ± 0.04,0.3488 ± 0.1086


Representative run:


,Model,Representative_Seed,Representative_F1_macro,Five_Run_Mean_F1_macro,Model_Path
0,DNN,42,0.664004,0.661224,/kaggle/working/CICIoT2023_baseline_without_FA...


All results saved to: /kaggle/working/CICIoT2023_baseline_without_FA_dnn_5_seed_results


In [22]:
import shutil

# Compress the entire Kaggle working/output directory
shutil.make_archive(
    "/kaggle/working/Baseline_CICIoT2023_NoFA_NoOptuna_DNN_5_Seed_V2",
    "zip",
    "/kaggle/working"
)

print("Created: /kaggle/working/Baseline_CICIoT2023_NoFA_NoOptuna_DNN_5_Seed_V2.zip")


Created: /kaggle/working/Baseline_CICIoT2023_NoFA_NoOptuna_DNN_5_Seed_V2.zip


## 8. Regenerate reports and figures from saved results (no retraining)

This final cell can be run in a fresh session after the five seed folders already exist. It rebuilds aggregate macro/weighted reports, mean ± sample-SD tables, per-class summaries, per-seed loss/confusion-matrix figures, and the representative loss/confusion-matrix outputs from the saved CSV files. No model training is performed.


In [23]:
# ============================================================
# RESULTS-ONLY REGENERATION UTILITY — NO MODEL TRAINING
# ============================================================

from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Change only this path when regenerating results in a new session.
RESULTS_DIRECTORY_FOR_REGENERATION = Path(
    r"CICIoT2023_baseline_without_FA_dnn_5_seed_results"
)


def _fmt_mean_sd(mean_value, sd_value, decimals=6, percentage=False):
    scale = 100.0 if percentage else 1.0
    if pd.isna(mean_value):
        return "NA"
    if pd.isna(sd_value):
        return f"{mean_value * scale:.{decimals}f} ± NA"
    return (
        f"{mean_value * scale:.{decimals}f} ± "
        f"{sd_value * scale:.{decimals}f}"
    )


def _plot_confusion_from_csv(csv_path, output_stem):
    frame = pd.read_csv(csv_path, index_col=0)

    plt.figure(figsize=(6.8, 5.5))
    sns.heatmap(
        frame.values,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        xticklabels=frame.columns,
        yticklabels=frame.index,
    )
    plt.xlabel("Predicted label", fontsize=11)
    plt.ylabel("True label", fontsize=11)
    plt.xticks(rotation=45, ha="right", fontsize=10)
    plt.yticks(rotation=45, va="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{output_stem}.pdf", bbox_inches="tight")
    plt.savefig(f"{output_stem}.png", dpi=600, bbox_inches="tight")
    plt.close()


def _plot_loss_from_csv(csv_path, output_stem):
    history = pd.read_csv(csv_path)

    if "loss" not in history.columns:
        raise KeyError(f"'loss' was not found in {csv_path}")

    epochs = np.arange(1, len(history) + 1)

    plt.figure(figsize=(6.8, 5.0))
    plt.plot(epochs, history["loss"], label="Training Loss")

    if "val_loss" in history.columns:
        plt.plot(
            epochs,
            history["val_loss"],
            label="Validation Loss",
        )

    plt.xlabel("Epoch", fontsize=11)
    plt.ylabel("Loss", fontsize=11)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{output_stem}.pdf", bbox_inches="tight")
    plt.savefig(f"{output_stem}.png", dpi=600, bbox_inches="tight")
    plt.close()


def regenerate_saved_results(results_directory):
    results_directory = Path(results_directory)

    if not results_directory.exists():
        raise FileNotFoundError(
            f"Results directory not found: {results_directory}"
        )

    config_path = results_directory / "experiment_config.json"
    if not config_path.exists():
        raise FileNotFoundError(
            f"experiment_config.json not found: {config_path}"
        )

    with config_path.open("r", encoding="utf-8") as file:
        config = json.load(file)

    model_type = str(config["model"]).upper()
    arch_lower = model_type.lower()
    run_seeds = [int(seed) for seed in config["run_seeds"]]

    seed_dirs = [
        results_directory / f"{model_type}_Seed_{run_number}"
        for run_number in range(1, len(run_seeds) + 1)
    ]

    for directory in seed_dirs:
        if not directory.exists():
            raise FileNotFoundError(
                f"Missing seed result folder: {directory}"
            )

    scalar_frames = []
    macro_frames = []
    weighted_frames = []
    per_class_frames = []

    for directory in seed_dirs:
        scalar_frames.append(
            pd.read_csv(directory / "scalar_metrics.csv")
        )
        macro_frames.append(
            pd.read_csv(directory / "macro_metrics.csv")
        )
        weighted_frames.append(
            pd.read_csv(directory / "weighted_average_metrics.csv")
        )
        per_class_frames.append(
            pd.read_csv(directory / "per_class_metrics_and_fpr.csv")
        )

        _plot_loss_from_csv(
            directory / "training_history.csv",
            directory / "loss_curve",
        )

        _plot_confusion_from_csv(
            directory / "confusion_matrix.csv",
            directory / "confusion_matrix",
        )

    all_runs = pd.concat(scalar_frames, ignore_index=True)
    all_macro = pd.concat(macro_frames, ignore_index=True)
    all_weighted = pd.concat(weighted_frames, ignore_index=True)
    all_per_class = pd.concat(per_class_frames, ignore_index=True)

    all_runs.to_csv(
        results_directory / f"{arch_lower}_5_seed_run_results.csv",
        index=False,
    )
    all_macro.to_csv(
        results_directory / f"{arch_lower}_macro_metrics_5_seed_results.csv",
        index=False,
    )
    all_weighted.to_csv(
        results_directory
        / f"{arch_lower}_weighted_average_metrics_5_seed_results.csv",
        index=False,
    )
    all_per_class.to_csv(
        results_directory
        / f"{arch_lower}_per_class_5_seed_run_results.csv",
        index=False,
    )

    # Overall five-run mean ± sample SD.
    identifier_columns = {"Model", "Run", "Seed"}
    metric_columns = [
        column
        for column in all_runs.columns
        if column not in identifier_columns
        and pd.api.types.is_numeric_dtype(all_runs[column])
    ]

    summary_rows = []
    for metric in metric_columns:
        values = pd.to_numeric(all_runs[metric], errors="coerce")
        mean_value = float(values.mean())
        sd_value = float(values.std(ddof=1))
        summary_rows.append({
            "Model": model_type,
            "Metric": metric,
            "Mean": mean_value,
            "SD": sd_value,
            "N": int(values.notna().sum()),
            "Mean ± SD": _fmt_mean_sd(mean_value, sd_value),
        })

    overall_summary = pd.DataFrame(summary_rows)
    overall_summary.to_csv(
        results_directory / "five_run_mean_sd_long.csv",
        index=False,
    )
    overall_summary[["Metric", "Mean ± SD"]].to_csv(
        results_directory / "five_run_mean_sd_report_table.csv",
        index=False,
    )

    def subset_summary(frame, metric_names):
        rows = []
        for metric in metric_names:
            values = pd.to_numeric(frame[metric], errors="coerce")
            mean_value = float(values.mean())
            sd_value = float(values.std(ddof=1))
            rows.append({
                "Model": model_type,
                "Metric": metric,
                "Mean": mean_value,
                "SD": sd_value,
                "N": int(values.notna().sum()),
                "Mean ± SD": _fmt_mean_sd(mean_value, sd_value),
            })
        return pd.DataFrame(rows)

    macro_summary = subset_summary(
        all_macro,
        [
            "Precision_macro",
            "Recall_macro",
            "F1_macro",
            "ROC_AUC_macro_OvR",
            "FPR_macro",
        ],
    )

    weighted_summary = subset_summary(
        all_weighted,
        [
            "Precision_weighted",
            "Recall_weighted",
            "F1_weighted",
            "FPR_weighted",
        ],
    )

    macro_summary.to_csv(
        results_directory / "macro_metrics_5_seed_mean_sd.csv",
        index=False,
    )
    weighted_summary.to_csv(
        results_directory
        / "weighted_average_metrics_5_seed_mean_sd.csv",
        index=False,
    )

    # Journal-ready macro and weighted tables.
    accuracy_mean = float(all_runs["Accuracy"].mean())
    accuracy_sd = float(all_runs["Accuracy"].std(ddof=1))

    journal_macro = pd.DataFrame([{
        "Model": model_type,
        "Accuracy (%)": _fmt_mean_sd(
            accuracy_mean, accuracy_sd, decimals=2, percentage=True
        ),
        "Recall (macro %)": _fmt_mean_sd(
            all_runs["Recall_macro"].mean(),
            all_runs["Recall_macro"].std(ddof=1),
            decimals=2,
            percentage=True,
        ),
        "Precision (macro %)": _fmt_mean_sd(
            all_runs["Precision_macro"].mean(),
            all_runs["Precision_macro"].std(ddof=1),
            decimals=2,
            percentage=True,
        ),
        "F1-score (macro %)": _fmt_mean_sd(
            all_runs["F1_macro"].mean(),
            all_runs["F1_macro"].std(ddof=1),
            decimals=2,
            percentage=True,
        ),
        "FPR (macro %)": _fmt_mean_sd(
            all_runs["FPR_macro"].mean(),
            all_runs["FPR_macro"].std(ddof=1),
            decimals=4,
            percentage=True,
        ),
        "ROC-AUC (macro, OvR)": _fmt_mean_sd(
            all_runs["ROC_AUC_macro_OvR"].mean(),
            all_runs["ROC_AUC_macro_OvR"].std(ddof=1),
            decimals=6,
        ),
    }])

    journal_weighted = pd.DataFrame([{
        "Model": model_type,
        "Accuracy (%)": _fmt_mean_sd(
            accuracy_mean, accuracy_sd, decimals=2, percentage=True
        ),
        "Recall (weighted %)": _fmt_mean_sd(
            all_runs["Recall_weighted"].mean(),
            all_runs["Recall_weighted"].std(ddof=1),
            decimals=2,
            percentage=True,
        ),
        "Precision (weighted %)": _fmt_mean_sd(
            all_runs["Precision_weighted"].mean(),
            all_runs["Precision_weighted"].std(ddof=1),
            decimals=2,
            percentage=True,
        ),
        "F1-score (weighted %)": _fmt_mean_sd(
            all_runs["F1_weighted"].mean(),
            all_runs["F1_weighted"].std(ddof=1),
            decimals=2,
            percentage=True,
        ),
        "FPR (weighted %)": _fmt_mean_sd(
            all_runs["FPR_weighted"].mean(),
            all_runs["FPR_weighted"].std(ddof=1),
            decimals=4,
            percentage=True,
        ),
    }])

    journal_macro.to_csv(
        results_directory / "journal_macro_mean_sd_table.csv",
        index=False,
    )
    journal_weighted.to_csv(
        results_directory / "journal_weighted_mean_sd_table.csv",
        index=False,
    )

    # Per-class five-run mean ± SD.
    per_class_rows = []

    for class_name, class_frame in all_per_class.groupby(
        "Class",
        sort=False,
    ):
        row = {
            "Model": model_type,
            "Class": class_name,
            "Support": int(class_frame["Support"].iloc[0]),
            "N": int(len(class_frame)),
        }

        for metric in ["Precision", "Recall", "F1", "FPR"]:
            mean_value = float(class_frame[metric].mean())
            sd_value = float(class_frame[metric].std(ddof=1))

            row[f"{metric}_Mean"] = mean_value
            row[f"{metric}_SD"] = sd_value
            row[f"{metric}_Mean ± SD"] = _fmt_mean_sd(
                mean_value,
                sd_value,
            )

        per_class_rows.append(row)

    pd.DataFrame(per_class_rows).to_csv(
        results_directory / "per_class_five_run_mean_sd.csv",
        index=False,
    )

    # Representative run: macro F1 closest to five-run mean.
    mean_macro_f1 = float(all_runs["F1_macro"].mean())
    representative_index = (
        all_runs["F1_macro"] - mean_macro_f1
    ).abs().idxmin()

    representative_row = all_runs.loc[representative_index]
    representative_run = int(representative_row["Run"])
    representative_seed = int(representative_row["Seed"])

    representative_dir = (
        results_directory
        / f"{model_type}_Seed_{representative_run}"
    )

    representative_cm_csv = (
        representative_dir / "confusion_matrix.csv"
    )

    representative_cm_stem = (
        results_directory
        / f"{arch_lower}_representative_confusion_matrix"
    )

    shutil.copyfile(
        representative_cm_csv,
        f"{representative_cm_stem}.csv",
    )

    _plot_confusion_from_csv(
        representative_cm_csv,
        representative_cm_stem,
    )

    shutil.copyfile(
        representative_dir / "classification_report.csv",
        results_directory
        / f"{arch_lower}_representative_classification_report.csv",
    )

    representative_history = (
        representative_dir / "training_history.csv"
    )

    shutil.copyfile(
        representative_history,
        results_directory
        / f"{arch_lower}_representative_training_history.csv",
    )

    _plot_loss_from_csv(
        representative_history,
        results_directory
        / f"{arch_lower}_representative_loss_curve",
    )

    model_files = list(
        representative_dir.glob(f"{arch_lower}_seed_*.keras")
    )
    model_path = (
        str(model_files[0])
        if model_files
        else "Saved model file not found"
    )

    representative_runs = pd.DataFrame([{
        "Model": model_type,
        "Representative_Run": representative_run,
        "Representative_Seed": representative_seed,
        "Representative_F1_macro": float(
            representative_row["F1_macro"]
        ),
        "Five_Run_Mean_F1_macro": mean_macro_f1,
        "Model_Path": model_path,
    }])

    representative_runs.to_csv(
        results_directory / "representative_runs.csv",
        index=False,
    )

    print("Results regenerated successfully without retraining.")
    print("Results directory:", results_directory)
    print(representative_runs.to_string(index=False))

    return {
        "all_run_results": all_runs,
        "macro_summary": macro_summary,
        "weighted_summary": weighted_summary,
        "journal_macro": journal_macro,
        "journal_weighted": journal_weighted,
        "representative_runs": representative_runs,
    }


regenerated_results = regenerate_saved_results(
    RESULTS_DIRECTORY_FOR_REGENERATION
)

Results regenerated successfully without retraining.
Results directory: CICIoT2023_baseline_without_FA_dnn_5_seed_results
Model  Representative_Run  Representative_Seed  Representative_F1_macro  Five_Run_Mean_F1_macro                                                                     Model_Path
  DNN                   3                   44                 0.666896                0.666028 CICIoT2023_baseline_without_FA_dnn_5_seed_results/DNN_Seed_3/dnn_seed_44.keras
